In [146]:
import pandas as pd
from pathlib import Path

In [172]:
# variables
lables_list = ["l1", "l2", "l3", "l4", "l5", "l6"]
eval_run_name= "l1-prompt-v10"
# eval_run_name= "2026-06-12"
labels = "data/labels"
load_path = f"{labels}/eval/{eval_run_name}/data"


In [173]:
def load_wos_excels(input_dir: Path) -> pd.DataFrame:
    paths = sorted(
        [path for pattern in ("*.xls", "*.xlsx") for path in Path(input_dir).glob(pattern)]
    )
    if not paths:
        raise FileNotFoundError(f"No .xls or .xlsx files found in {input_dir}")

    frames = [pd.read_excel(path) for path in paths]
    return pd.concat(frames, ignore_index=True)

In [174]:
l1_df = load_wos_excels(f"{load_path}/l1")
# l2_df = load_wos_excels(f"{load_path}/l2")
# l3_df = load_wos_excels(f"{load_path}/l3")
# l4_df = load_wos_excels(f"{load_path}/l4")
# l5_df = load_wos_excels(f"{load_path}/l5")
# l6_df = load_wos_excels(f"{load_path}/l6")

In [175]:
def add_record_level_confusion(df: pd.DataFrame, true_col: str, pred_col: str, base_col: str) -> pd.DataFrame:
    """Add record-level TP/FP/FN counts and Jaccard score using set comparisons (case-insensitive).
    Rows with missing/NaN truth are skipped (metrics set to None for those rows).
    Accepts list-like values or stringified lists (e.g., '["A", "B"]') in the label columns."""
    missing = [col for col in (true_col, pred_col) if col not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in dataframe: {missing}")

    for col_name in (true_col, pred_col):
        if df[col_name].dtype != 'object':
            raise TypeError(f"Column '{col_name}' must be of dtype object with iterables of strings")

    def _to_label_set(value, col_name):
        import ast

        if isinstance(value, (list, tuple, set)):
            seq = list(value)
        elif isinstance(value, str):
            try:
                parsed = ast.literal_eval(value)
            except Exception:
                parsed = value

            if isinstance(parsed, str):
                seq = [parsed]
            elif isinstance(parsed, (list, tuple, set)):
                seq = list(parsed)
            else:
                raise TypeError(
                    f"Entries in column '{col_name}' must be list-like or stringified lists of strings, got {type(parsed)}"
                )
        elif pd.isna(value):
            seq = []
        else:
            raise TypeError(
                f"Entries in column '{col_name}' must be list-like or string/stringified list of strings, got {type(value)}"
            )

        if not all(isinstance(item, str) for item in seq):
            raise TypeError(f"All entries in column '{col_name}' must be strings")

        normalized = [item.lower() for item in seq]
        return set(normalized)

    valid_mask = ~df[true_col].isna()

    true_sets = df[true_col].apply(lambda v: _to_label_set(v, true_col))
    pred_sets = df[pred_col].apply(lambda v: _to_label_set(v, pred_col))

    tp_vals = [None] * len(df)
    fp_vals = [None] * len(df)
    fn_vals = [None] * len(df)
    jacc_vals = [None] * len(df)

    for idx, (t, p, keep) in enumerate(zip(true_sets, pred_sets, valid_mask)):
        if not keep:
            continue
        tp = len(t & p)
        fp = len(p - t)
        fn = len(t - p)
        union = t | p
        jaccard = tp / len(union) if union else 0.0

        tp_vals[idx] = tp
        fp_vals[idx] = fp
        fn_vals[idx] = fn
        jacc_vals[idx] = jaccard

    new_cols = {
        f"{base_col}_tp": tp_vals,
        f"{base_col}_fp": fp_vals,
        f"{base_col}_fn": fn_vals,
        f"{base_col}_jaccard": jacc_vals,
    }

    for col_name, values in new_cols.items():
        if col_name in df.columns:
            raise ValueError(f"Column '{col_name}' already exists on dataframe")
        df[col_name] = values

    return "Df updated with record-level confusion metrics. New cols added: " + ", ".join(new_cols.keys())


In [176]:
def compute_label_level_metrics(df: pd.DataFrame, true_col: str, pred_col: str) -> pd.DataFrame:
    """Compute TP/FP/FN and per-label precision/recall/F1/Jaccard (case-insensitive).
    Rows with missing/NaN truth are skipped. Adds a support column (tp+fn) and
    a weighted summary row (weights = support). Accepts list-like values or
    stringified lists (e.g., '["A", "B"]') in the label columns."""
    missing = [col for col in (true_col, pred_col) if col not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in dataframe: {missing}")

    for col_name in (true_col, pred_col):
        if df[col_name].dtype != 'object':
            raise TypeError(f"Column '{col_name}' must be of dtype object with iterables of strings")

    def _to_label_set(value, col_name):
        import ast

        if isinstance(value, (list, tuple, set)):
            seq = list(value)
        elif isinstance(value, str):
            try:
                parsed = ast.literal_eval(value)
            except Exception:
                parsed = value

            if isinstance(parsed, str):
                seq = [parsed]
            elif isinstance(parsed, (list, tuple, set)):
                seq = list(parsed)
            else:
                raise TypeError(
                    f"Entries in column '{col_name}' must be list-like or stringified lists of strings, got {type(parsed)}"
                )
        elif pd.isna(value):
            seq = []
        else:
            raise TypeError(
                f"Entries in column '{col_name}' must be list-like or string/stringified list of strings, got {type(value)}"
            )

        if not all(isinstance(item, str) for item in seq):
            raise TypeError(f"All entries in column '{col_name}' must be strings")

        normalized = [item.lower() for item in seq]
        return set(normalized)

    valid_mask = ~df[true_col].isna()
    df_valid = df.loc[valid_mask]

    if df_valid.empty:
        return pd.DataFrame(columns=["label", "tp", "fp", "fn", "precision", "recall", "f1", "jaccard", "support"])

    true_sets = df_valid[true_col].apply(lambda v: _to_label_set(v, true_col))
    pred_sets = df_valid[pred_col].apply(lambda v: _to_label_set(v, pred_col))

    labels = sorted(set().union(*true_sets, *pred_sets))

    rows = []
    for label in labels:
        tp = sum((label in t) and (label in p) for t, p in zip(true_sets, pred_sets))
        fp = sum((label in p) and (label not in t) for t, p in zip(true_sets, pred_sets))
        fn = sum((label in t) and (label not in p) for t, p in zip(true_sets, pred_sets))

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        jaccard = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
        support = tp + fn

        rows.append(
            {
                "label": label,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "jaccard": jaccard,
                "support": support,
            }
        )

    support_sum = sum(r["support"] for r in rows)
    if support_sum:
        weighted_precision = sum(r["precision"] * r["support"] for r in rows) / support_sum
        weighted_recall = sum(r["recall"] * r["support"] for r in rows) / support_sum
        weighted_f1 = sum(r["f1"] * r["support"] for r in rows) / support_sum
        weighted_jaccard = sum(r["jaccard"] * r["support"] for r in rows) / support_sum
    else:
        weighted_precision = weighted_recall = weighted_f1 = weighted_jaccard = 0.0

    rows.append(
        {
            "label": "_weighted",
            "tp": None,
            "fp": None,
            "fn": None,
            "precision": weighted_precision,
            "recall": weighted_recall,
            "f1": weighted_f1,
            "jaccard": weighted_jaccard,
            "support": support_sum,
        }
    )

    return pd.DataFrame(rows)


In [177]:
add_record_level_confusion(l1_df, "driver", "results", "driver")


'Df updated with record-level confusion metrics. New cols added: driver_tp, driver_fp, driver_fn, driver_jaccard'

In [178]:
l1_df_macro= compute_label_level_metrics(l1_df, "driver", "results")


In [179]:
l1_df_macro #v10

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,542.0,149.0,87.0,0.784370,0.861685,0.821212,0.696658,629
1,direct exploitation and resource extraction,795.0,98.0,183.0,0.890258,0.812883,0.849813,0.738848,978
2,invasive alien species,377.0,111.0,31.0,0.772541,0.924020,0.841518,0.726397,408
3,land/sea use change,783.0,259.0,207.0,0.751440,0.790909,0.770669,0.626902,990
4,not applicable,0.0,118.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1479.0,260.0,123.0,0.850489,0.923221,0.885364,0.794307,1602
6,_weighted,NaN,NaN,NaN,0.821716,0.863035,0.840528,0.727214,4607


In [171]:
l1_df_macro #v9

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,526.0,143.0,103.0,0.786248,0.836248,0.810478,0.681347,629
1,direct exploitation and resource extraction,765.0,89.0,213.0,0.895785,0.782209,0.835153,0.716963,978
2,invasive alien species,375.0,114.0,33.0,0.766871,0.919118,0.836120,0.718391,408
3,land/sea use change,821.0,322.0,169.0,0.718285,0.829293,0.769808,0.625762,990
4,not applicable,0.0,113.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1490.0,265.0,112.0,0.849003,0.930087,0.887697,0.798072,1602
6,_weighted,NaN,NaN,NaN,0.815002,0.863252,0.836099,0.720833,4607


In [155]:
l1_df_macro #v8

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,526.0,138.0,103.0,0.792169,0.836248,0.813612,0.685789,629
1,direct exploitation and resource extraction,757.0,102.0,221.0,0.881257,0.774029,0.824170,0.700926,978
2,invasive alien species,377.0,126.0,31.0,0.749503,0.924020,0.827662,0.705993,408
3,land/sea use change,781.0,297.0,209.0,0.724490,0.788889,0.755319,0.606838,990
4,not applicable,0.0,106.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1489.0,283.0,113.0,0.840293,0.929463,0.882632,0.789920,1602
6,_weighted,NaN,NaN,NaN,0.809493,0.853050,0.828571,0.710035,4607


In [145]:
l1_df_macro #v4?

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,551.0,168.0,78.0,0.766342,0.875994,0.817507,0.691343,629
1,direct exploitation and resource extraction,784.0,124.0,194.0,0.863436,0.801636,0.831389,0.711434,978
2,invasive alien species,386.0,145.0,22.0,0.726930,0.946078,0.822151,0.698011,408
3,land/sea use change,774.0,279.0,216.0,0.735043,0.781818,0.757709,0.609929,990
4,not applicable,0.0,77.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1493.0,294.0,109.0,0.835478,0.931960,0.881086,0.787447,1602
6,_weighted,NaN,NaN,NaN,0.800778,0.865639,0.830124,0.712122,4607


In [118]:
l1_df_macro #v7

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,554.0,180.0,75.0,0.754768,0.880763,0.812913,0.684796,629
1,direct exploitation and resource extraction,764.0,119.0,214.0,0.865232,0.781186,0.821064,0.696445,978
2,invasive alien species,388.0,165.0,20.0,0.701627,0.950980,0.807492,0.677138,408
3,land/sea use change,830.0,438.0,160.0,0.654574,0.838384,0.735164,0.581232,990
4,not applicable,0.0,58.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1475.0,293.0,127.0,0.834276,0.920724,0.875371,0.778364,1602
6,_weighted,NaN,NaN,NaN,0.779629,0.870632,0.819174,0.696872,4607


In [109]:
l1_df_macro #v6

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,543.0,167.0,86.0,0.764789,0.863275,0.811053,0.682161,629
1,direct exploitation and resource extraction,795.0,149.0,183.0,0.842161,0.812883,0.827263,0.705413,978
2,invasive alien species,384.0,132.0,24.0,0.744186,0.941176,0.831169,0.711111,408
3,land/sea use change,684.0,151.0,306.0,0.819162,0.690909,0.749589,0.599474,990
4,not applicable,0.0,138.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1488.0,283.0,114.0,0.840203,0.928839,0.882301,0.789390,1602
6,_weighted,NaN,NaN,NaN,0.817297,0.845236,0.827843,0.709179,4607


In [101]:
l1_df_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,552.0,182.0,77.0,0.752044,0.877583,0.809978,0.680641,629
1,direct exploitation and resource extraction,762.0,112.0,216.0,0.871854,0.779141,0.822894,0.699083,978
2,invasive alien species,391.0,145.0,17.0,0.729478,0.958333,0.828390,0.707052,408
3,land/sea use change,822.0,412.0,168.0,0.666126,0.830303,0.739209,0.586305,990
4,not applicable,0.0,63.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1494.0,301.0,108.0,0.832312,0.932584,0.879600,0.785076,1602
6,_weighted,NaN,NaN,NaN,0.784928,0.872802,0.823352,0.702939,4607


In [66]:
l1_df_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,551.0,168.0,78.0,0.766342,0.875994,0.817507,0.691343,629
1,direct exploitation and resource extraction,784.0,124.0,194.0,0.863436,0.801636,0.831389,0.711434,978
2,invasive alien species,386.0,145.0,22.0,0.726930,0.946078,0.822151,0.698011,408
3,land/sea use change,774.0,279.0,216.0,0.735043,0.781818,0.757709,0.609929,990
4,not applicable,0.0,77.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1493.0,294.0,109.0,0.835478,0.931960,0.881086,0.787447,1602
6,_weighted,NaN,NaN,NaN,0.800778,0.865639,0.830124,0.712122,4607


In [56]:
l1_df_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,climate change,546.0,166.0,83.0,0.766854,0.868045,0.814318,0.686792,629
1,direct exploitation and resource extraction,788.0,131.0,190.0,0.857454,0.805726,0.830785,0.710550,978
2,invasive alien species,376.0,120.0,32.0,0.758065,0.921569,0.831858,0.712121,408
3,land/sea use change,773.0,253.0,217.0,0.753411,0.780808,0.766865,0.621883,990
4,not applicable,0.0,102.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,pollution,1483.0,285.0,119.0,0.838801,0.925718,0.880119,0.785904,1602
6,_weighted,NaN,NaN,NaN,0.807438,0.860864,0.832051,0.714595,4607


### Others

In [90]:
l2_df_threat_l0_macro= compute_label_level_metrics(l2_df, "threats_l0", "threat_l0_results")
l2_df_threat_l1_macro= compute_label_level_metrics(l2_df, "threats_l1", "threat_l1_results")

l3_df_region_macro= compute_label_level_metrics(l3_df, "region", "regions")
l3_df_sub_region_macro= compute_label_level_metrics(l3_df, "sub-region", "subregions")
l3_df_country_macro= compute_label_level_metrics(l3_df, "country", "countries_iso3")

l4_df_realms_macro= compute_label_level_metrics(l4_df, "realm", "realm_results")
l4_df_biome_macro= compute_label_level_metrics(l4_df, "biome", "biome_results")

l5_df_study_macro= compute_label_level_metrics(l5_df, "study_design", "pred_study_design")

In [91]:
l2_df_threat_l0_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,agriculture & aquaculture,22.0,12.0,48.0,0.647059,0.314286,0.423077,0.268293,70
1,biological resource use,100.0,6.0,149.0,0.943396,0.401606,0.563380,0.392157,249
2,climate change & severe weather,15.0,3.0,13.0,0.833333,0.535714,0.652174,0.483871,28
3,climate change & severe weather,0.0,0.0,1.0,0.000000,0.000000,0.000000,0.000000,1
4,energy production & mining,3.0,5.0,9.0,0.375000,0.250000,0.300000,0.176471,12
5,geological events,0.0,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0
6,human intrusions & disturbance,11.0,11.0,17.0,0.500000,0.392857,0.440000,0.282051,28
7,"invasive & other problematic species, genes & ...",88.0,5.0,69.0,0.946237,0.560510,0.704000,0.543210,157
8,natural system modifications,23.0,34.0,59.0,0.403509,0.280488,0.330935,0.198276,82
9,not applicable,0.0,110.0,0.0,0.000000,0.000000,0.000000,0.000000,0


In [110]:
l2_df_threat_l1_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,agricultural & forestry effluents,16.0,11.0,74.0,0.592593,0.177778,0.273504,0.158416,90
1,agriculture & aquaculture,0.0,28.0,0.0,0.000000,0.000000,0.000000,0.000000,0
2,air-borne pollutants,6.0,4.0,8.0,0.600000,0.428571,0.500000,0.333333,14
3,annual & perennial non-timber crops,39.0,31.0,116.0,0.557143,0.251613,0.346667,0.209677,155
4,biological resource use,0.0,17.0,0.0,0.000000,0.000000,0.000000,0.000000,0
5,climate change & severe weather,0.0,4.0,0.0,0.000000,0.000000,0.000000,0.000000,0
6,commercial & industrial areas,2.0,2.0,74.0,0.500000,0.026316,0.050000,0.025641,76
7,dams & water management/use,15.0,6.0,63.0,0.714286,0.192308,0.303030,0.178571,78
8,diseases of unknown cause,1.0,0.0,2.0,1.000000,0.333333,0.500000,0.333333,3
9,domestic & urban waste water,5.0,28.0,26.0,0.151515,0.161290,0.156250,0.084746,31


In [45]:
l3_df_region_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,africa,57.0,3.0,2.0,0.950000,0.966102,0.957983,0.919355,59
1,africa/asia-pacific,0.0,0.0,1.0,0.000000,0.000000,0.000000,0.000000,1
2,all region,0.0,56.0,0.0,0.000000,0.000000,0.000000,0.000000,0
3,all regions,0.0,0.0,31.0,0.000000,0.000000,0.000000,0.000000,31
4,americas,373.0,8.0,22.0,0.979003,0.944304,0.961340,0.925558,395
5,americas/africa/asia-pacific,0.0,0.0,3.0,0.000000,0.000000,0.000000,0.000000,3
6,americas/asia-pacific,0.0,0.0,1.0,0.000000,0.000000,0.000000,0.000000,1
7,asia and the pacific,53.0,51.0,2.0,0.509615,0.963636,0.666667,0.500000,55
8,asia-pacific,0.0,0.0,37.0,0.000000,0.000000,0.000000,0.000000,37
9,europe and central asia,281.0,5.0,18.0,0.982517,0.939799,0.960684,0.924342,299


In [74]:
l3_df_sub_region_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,central and western europe,0.0,1.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,mesoamerica,0.0,1.0,0.0,0.0,0.000000,0.000000,0.000000,0
2,north america,278.0,0.0,22.0,1.0,0.926667,0.961938,0.926667,300
3,north-east asia,0.0,1.0,0.0,0.0,0.000000,0.000000,0.000000,0
4,oceania,13.0,0.0,1.0,1.0,0.928571,0.962963,0.928571,14
5,south america,2.0,0.0,1.0,1.0,0.666667,0.800000,0.666667,3
6,south asia,8.0,0.0,0.0,1.0,1.000000,1.000000,1.000000,8
7,_weighted,NaN,NaN,NaN,1.0,0.926154,0.961424,0.926154,325


In [86]:
l4_df_realms_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,,0.0,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0
1,all realms,1.0,6.0,8.0,0.142857,0.111111,0.125000,0.066667,9
2,freshwater,105.0,67.0,37.0,0.610465,0.739437,0.668790,0.502392,142
3,freshwater-marine,2.0,21.0,4.0,0.086957,0.333333,0.137931,0.074074,6
4,freshwater-terrestrial,1.0,36.0,7.0,0.027027,0.125000,0.044444,0.022727,8
5,marine,283.0,4.0,58.0,0.986063,0.829912,0.901274,0.820290,341
6,marine-terrestrial,0.0,47.0,1.0,0.000000,0.000000,0.000000,0.000000,1
7,not applicable,0.0,22.0,0.0,0.000000,0.000000,0.000000,0.000000,0
8,subterranean,0.0,3.0,0.0,0.000000,0.000000,0.000000,0.000000,0
9,terrestrial,1646.0,15.0,135.0,0.990969,0.924200,0.956421,0.916481,1781


In [87]:
l4_df_biome_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,deserts and semi-deserts,0.0,0.0,1.0,0.000000,0.000000,0.000000,0.000000,1
1,savannas and grasslands,2.0,1.0,1.0,0.666667,0.666667,0.666667,0.500000,3
2,temperate-boreal forests & woodlands,0.0,2.0,0.0,0.000000,0.000000,0.000000,0.000000,0
3,tropical-subtropical forests,43.0,0.0,1.0,1.000000,0.977273,0.988506,0.977273,44
4,_weighted,NaN,NaN,NaN,0.958333,0.937500,0.947797,0.927083,48


In [89]:
l5_df_study_macro

,label,tp,fp,fn,precision,recall,f1,jaccard,support
0,experimental,718.0,130.0,115.0,0.846698,0.861945,0.854253,0.745587,833
1,modelling,400.0,404.0,74.0,0.497512,0.843882,0.625978,0.455581,474
2,observational,2968.0,180.0,672.0,0.942821,0.815385,0.874484,0.776963,3640
3,quasiexperimental,0.0,45.0,0.0,0.000000,0.000000,0.000000,0.000000,0
4,review,109.0,52.0,89.0,0.677019,0.550505,0.607242,0.436000,198
5,unclear,0.0,45.0,0.0,0.000000,0.000000,0.000000,0.000000,0
6,_weighted,NaN,NaN,NaN,0.876003,0.815355,0.838030,0.729153,5145


### Save


In [48]:
l3_df.region.unique()

array(['["Asia and the Pacific"]', '["Africa"]', '["Americas"]', nan,
       '["All regions"]', '["Asia-Pacific"]',
       '["Europe and Central Asia"]', '["Americas/Africa/Asia-Pacific"]',
       '["Africa/Asia-Pacific"]', '["Unclear or not specified"]',
       '["Americas/Asia-Pacific"]'], dtype=object)

In [162]:
l1_df.to_excel("l1_eval_output.xlsx", index=False)

In [163]:
l1_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4258 entries, 0 to 4257
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                4258 non-null   object 
 1   custom_id         4258 non-null   object 
 2   response          4258 non-null   object 
 3   error             0 non-null      float64
 4   response_json     4258 non-null   object 
 5   output_text       4258 non-null   object 
 6   input_tokens      4258 non-null   int64  
 7   cached_tokens     4258 non-null   int64  
 8   output_tokens     4258 non-null   int64  
 9   reasoning_tokens  4258 non-null   int64  
 10  total_tokens      4258 non-null   int64  
 11  results           4258 non-null   object 
 12  Article Title     4258 non-null   object 
 13  Abstract          4232 non-null   object 
 14  DOI               4245 non-null   object 
 15  source            4258 non-null   object 
 16  driver            4225 non-null   object 
